# AfyaFlow Pwani: Gemma 4 Stock-Out Early Warning Demo

**Track 3 — Smart Health:** turn a multilingual facility stock report into a verified inventory record, deterministic stock-out risk, transfer recommendation, and bilingual handoff.

This notebook is **self-contained**. It does not clone GitHub or import the local `src/` package. Attach only:

1. **GPU** accelerator
2. Your **Keras Gemma 4** model Input (`gemma4` / V2)

Then **Run All**.

## 1. Install KerasHub and set backend

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")

%pip install -q -U "keras>=3.8" "keras-hub>=0.21"

import keras
import keras_hub

print("keras", keras.__version__)
print("keras_hub", keras_hub.__version__)
print("backend", keras.config.backend())

## 2. AfyaFlow core (embedded)

Schemas, deterministic risk engine, approved tools, and synthetic demo data — all in-notebook so GitHub linking of this file alone is enough.

In [ ]:
from __future__ import annotations

import json
import math
import re
from dataclasses import asdict, dataclass, field
from datetime import date
from pathlib import Path
from typing import Any, Literal

RiskLevel = Literal["green", "amber", "red"]
SourceType = Literal["text", "image", "audio"]

RED_THRESHOLD_DAYS = 3
AMBER_THRESHOLD_DAYS = 7
SURPLUS_THRESHOLD_DAYS = 14
TRANSFER_BUFFER_DAYS = 7


@dataclass(frozen=True)
class StockReport:
    facility: str
    item: str
    balance_units: int
    average_daily_use: int
    expiry_date: date | None
    report_date: date
    source_type: SourceType
    source_language: str
    confidence: float
    patient_footfall_today: int | None = None
    notes: str = ""

    def __post_init__(self) -> None:
        if not self.facility.strip() or not self.item.strip():
            raise ValueError("facility and item are required")
        if self.balance_units < 0 or self.average_daily_use <= 0:
            raise ValueError("invalid balance_units or average_daily_use")
        if not 0 <= self.confidence <= 1:
            raise ValueError("confidence must be between 0 and 1")


@dataclass(frozen=True)
class FacilityInventory:
    facility: str
    level: str
    county: str
    item: str
    balance_units: int
    average_daily_use: int
    distance_km: float


@dataclass(frozen=True)
class TransferOption:
    facility: str
    item: str
    surplus_units: int
    days_of_stock_after_transfer: float
    distance_km: float
    reason: str


@dataclass(frozen=True)
class StockRisk:
    days_of_stock: float
    adjusted_daily_use: int
    level: RiskLevel
    reason: str
    transfer_rank: list[TransferOption] = field(default_factory=list)


def estimate_adjusted_daily_use(report: StockReport) -> int:
    if report.patient_footfall_today is None:
        return report.average_daily_use
    if report.patient_footfall_today > report.average_daily_use * 2:
        return max(1, math.ceil(report.average_daily_use * 1.25))
    return report.average_daily_use


def rank_transfer_options(
    report: StockReport,
    inventory: list[FacilityInventory],
    max_options: int = 3,
) -> list[TransferOption]:
    matches: list[TransferOption] = []
    for candidate in inventory:
        if candidate.item.lower() != report.item.lower():
            continue
        candidate_days = candidate.balance_units / candidate.average_daily_use
        if candidate_days <= SURPLUS_THRESHOLD_DAYS:
            continue
        surplus_units = math.floor(
            candidate.balance_units - (candidate.average_daily_use * TRANSFER_BUFFER_DAYS)
        )
        if surplus_units <= 0:
            continue
        matches.append(
            TransferOption(
                facility=candidate.facility,
                item=candidate.item,
                surplus_units=surplus_units,
                days_of_stock_after_transfer=round(
                    (candidate.balance_units - surplus_units) / candidate.average_daily_use, 2
                ),
                distance_km=candidate.distance_km,
                reason=(
                    f"{candidate.facility} has surplus {candidate.item} "
                    f"and is {candidate.distance_km:g} km away."
                ),
            )
        )
    return sorted(matches, key=lambda o: (o.distance_km, -o.surplus_units))[:max_options]


def calculate_stock_risk_with_transfers(
    report: StockReport,
    inventory: list[FacilityInventory],
) -> StockRisk:
    adjusted = estimate_adjusted_daily_use(report)
    days = report.balance_units / adjusted
    if days < RED_THRESHOLD_DAYS:
        level: RiskLevel = "red"
        reason = "Less than 3 days of stock remaining; immediate action is needed."
    elif days < AMBER_THRESHOLD_DAYS:
        level = "amber"
        reason = "Between 3 and 7 days of stock remaining; plan replenishment now."
    else:
        level = "green"
        reason = "At least 7 days of stock remaining; continue routine monitoring."
    return StockRisk(
        days_of_stock=round(days, 2),
        adjusted_daily_use=adjusted,
        level=level,
        reason=reason,
        transfer_rank=rank_transfer_options(report, inventory),
    )


def draft_handoff_message(report: StockReport, risk: StockRisk, language: str = "sw") -> str:
    first = risk.transfer_rank[0] if risk.transfer_rank else None
    if language.startswith("sw"):
        destination = first.facility if first else "ghala ya kaunti"
        return (
            f"Tahadhari: {report.facility} ina {report.balance_units} za {report.item}, "
            f"takriban siku {risk.days_of_stock}. Tafadhali hakiki na panga usaidizi "
            f"kutoka {destination}."
        )
    destination = first.facility if first else "the district store"
    return (
        f"Alert: {report.facility} has {report.balance_units} {report.item} remaining, "
        f"about {risk.days_of_stock} days of stock. Please verify and coordinate support "
        f"from {destination}."
    )


def build_extraction_prompt(raw_input: str, source_type: str = "text") -> str:
    return f"""You are AfyaFlow Pwani, a stock-report extraction assistant.
Extract only inventory-management facts from the user's {source_type} input.
Return strict JSON with these keys:
facility, item, balance_units, average_daily_use, expiry_date, report_date,
source_type, source_language, confidence, patient_footfall_today, notes.
Use null when a field is absent. Do not invent patient data or clinical advice.

Input:
{raw_input}
""".strip()


def parse_stock_report_payload(payload: dict[str, Any]) -> StockReport:
    def parse_optional_date(value: Any) -> date | None:
        if value in {None, ""}:
            return None
        return date.fromisoformat(str(value)[:10])

    return StockReport(
        facility=str(payload["facility"]),
        item=str(payload["item"]),
        balance_units=int(payload["balance_units"]),
        average_daily_use=int(payload["average_daily_use"]),
        expiry_date=parse_optional_date(payload.get("expiry_date")),
        report_date=date.fromisoformat(str(payload["report_date"])[:10]),
        source_type=payload.get("source_type", "text"),
        source_language=str(payload.get("source_language", "unknown")),
        confidence=float(payload.get("confidence", 0.8)),
        patient_footfall_today=(
            None
            if payload.get("patient_footfall_today") in {None, ""}
            else int(payload["patient_footfall_today"])
        ),
        notes=str(payload.get("notes", "")),
    )


print("AfyaFlow core loaded.")

In [ ]:
INVENTORY_RAW = [
    {"facility": "Old Town Health Centre", "level": "PHC", "county": "Mombasa", "item": "ORS sachets", "balance_units": 12, "average_daily_use": 6, "distance_km": 0},
    {"facility": "Tudor Community Health Centre", "level": "CHC", "county": "Mombasa", "item": "ORS sachets", "balance_units": 180, "average_daily_use": 9, "distance_km": 4.2},
    {"facility": "Kisauni Primary Health Centre", "level": "PHC", "county": "Mombasa", "item": "ORS sachets", "balance_units": 95, "average_daily_use": 8, "distance_km": 7.8},
    {"facility": "Likoni Community Health Centre", "level": "CHC", "county": "Mombasa", "item": "Malaria RDT kits", "balance_units": 28, "average_daily_use": 12, "distance_km": 6.5},
    {"facility": "Changamwe Primary Health Centre", "level": "PHC", "county": "Mombasa", "item": "Malaria RDT kits", "balance_units": 260, "average_daily_use": 11, "distance_km": 8.1},
    {"facility": "Mtwapa Health Centre", "level": "PHC", "county": "Kilifi", "item": "Amoxicillin 250mg capsules", "balance_units": 36, "average_daily_use": 18, "distance_km": 14.4},
    {"facility": "Kilifi County Referral Outpatient Store", "level": "CHC", "county": "Kilifi", "item": "Amoxicillin 250mg capsules", "balance_units": 520, "average_daily_use": 22, "distance_km": 52.3},
]

EXAMPLES = [
    {
        "id": "golden-ors-low-stock",
        "raw_input": "Old Town Health Centre tuko na ORS sachets 12 tu, matumizi ni around 6 kwa siku, report ya 31 July 2026.",
        "expected": {"facility": "Old Town Health Centre", "item": "ORS sachets", "risk_level": "red"},
    },
    {
        "id": "malaria-rdt-low-stock",
        "raw_input": "Likoni Community Health Centre has 28 malaria RDT kits left and uses about 12 per day. Please flag this report for July 31 2026.",
        "expected": {"facility": "Likoni Community Health Centre", "item": "Malaria RDT kits", "risk_level": "red"},
    },
    {
        "id": "amoxicillin-amber",
        "raw_input": "Mtwapa Health Centre reports 82 amoxicillin 250mg capsules remaining with average consumption of 14 per day. Report date 2026-07-31.",
        "expected": {"facility": "Mtwapa Health Centre", "item": "Amoxicillin 250mg capsules", "risk_level": "amber"},
    },
]

inventory = [FacilityInventory(**row) for row in INVENTORY_RAW]
print(f"Loaded {len(inventory)} synthetic inventory rows and {len(EXAMPLES)} demo cases.")

## 3. Load attached Gemma 4 model

Finds your Kaggle model Input automatically (looks for `config.json` + `tokenizer.json` + `model.weights.json` under `/kaggle/input`).

In [ ]:
def find_gemma4_preset(search_roots: list[Path] | None = None) -> Path:
    roots = search_roots or [Path("/kaggle/input"), Path.cwd() / "models", Path.cwd()]
    preferred = ("gemma4_instruct_2b", "gemma4_instruct_4b", "gemma4_instruct", "gemma4")
    candidates: list[Path] = []

    for root in roots:
        if not root.exists():
            continue
        for config_path in root.rglob("config.json"):
            preset_dir = config_path.parent
            if all((preset_dir / name).exists() for name in ("model.weights.json", "tokenizer.json", "task.json")):
                candidates.append(preset_dir)

    if not candidates:
        raise FileNotFoundError(
            "No Gemma 4 Keras preset found under /kaggle/input. "
            "Add the keras/gemma4 model as a notebook Input, then re-run."
        )

    def rank(path: Path) -> tuple[int, int, str]:
        text = str(path).lower()
        prefer = next((i for i, marker in enumerate(preferred) if marker in text), len(preferred))
        return (prefer, 0 if "instruct" in text else 1, str(path))

    candidates = sorted(set(candidates), key=rank)
    print("Found Gemma preset candidates:")
    for path in candidates:
        print(" -", path)
    return candidates[0]


GEMMA_PRESET = find_gemma4_preset()
print("Using preset:", GEMMA_PRESET)
print("Preset files:", sorted(p.name for p in GEMMA_PRESET.iterdir())[:15])

In [ ]:
print("Loading Gemma 4 (this can take several minutes on first load)...")
gemma_lm = keras_hub.models.Gemma4CausalLM.from_preset(
    str(GEMMA_PRESET),
    dtype="bfloat16",
)
print("Loaded:", type(gemma_lm).__name__)

## 4. Gemma runtime → structured stock report

In [ ]:
class KaggleGemmaRuntime:
    """Ask Gemma 4 for JSON, then parse into a dict for AfyaFlow validation."""

    def __init__(self, model, *, max_length: int = 768) -> None:
        self.model = model
        self.max_length = max_length
        self.last_raw_text = ""

    def _wrap_prompt(self, prompt: str) -> str:
        return (
            "<|turn>user\n"
            f"{prompt}\n"
            "Return ONLY a single JSON object. No markdown fences. No commentary.<turn|>\n"
            "<|turn>model\n"
        )

    def _strip_prompt(self, output: Any, prompt: str) -> str:
        if isinstance(output, list):
            output = output[0]
        text = str(output)
        if text.startswith(prompt):
            return text[len(prompt):]
        for marker in ("<|turn>model\n", "<start_of_turn>model\n"):
            index = text.rfind(marker)
            if index != -1:
                return text[index + len(marker):]
        return text

    def _extract_json_object(self, text: str) -> dict[str, Any]:
        cleaned = text.strip()
        fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", cleaned, flags=re.DOTALL)
        if fenced:
            cleaned = fenced.group(1)
        else:
            match = re.search(r"(\{.*\})", cleaned, flags=re.DOTALL)
            if not match:
                raise ValueError(f"No JSON object found in model output: {text[:400]!r}")
            cleaned = match.group(1)
        payload = json.loads(cleaned)
        if not isinstance(payload, dict):
            raise ValueError("Model JSON must be an object")
        return payload

    def generate_json(self, prompt: str) -> dict[str, Any]:
        wrapped = self._wrap_prompt(prompt)
        output = self.model.generate({"prompts": [wrapped]}, max_length=self.max_length)
        raw = self._strip_prompt(output, wrapped)
        self.last_raw_text = raw
        print("Gemma raw output:\n", raw[:1200])
        return self._extract_json_object(raw)


def extract_with_gemma(raw_input: str, runtime: KaggleGemmaRuntime, source_type: str = "text") -> StockReport:
    prompt = build_extraction_prompt(raw_input, source_type=source_type)
    payload = runtime.generate_json(prompt)
    payload.setdefault("source_type", source_type)
    if not payload.get("report_date"):
        payload["report_date"] = "2026-07-31"
    return parse_stock_report_payload(payload)


runtime = KaggleGemmaRuntime(gemma_lm)
print("Gemma runtime ready.")

## 5. Golden demo: report → Gemma extract → risk → transfer → handoff

In [ ]:
raw_input = EXAMPLES[0]["raw_input"]
print("Facility report:\n", raw_input)
print()
print(build_extraction_prompt(raw_input))

In [ ]:
print("Extracting with live Gemma 4...")
report = extract_with_gemma(raw_input, runtime)
asdict(report)

In [ ]:
risk = calculate_stock_risk_with_transfers(report, inventory)
handoff = draft_handoff_message(report, risk, language="sw")

{
    "facility": report.facility,
    "item": report.item,
    "balance_units": report.balance_units,
    "days_of_stock": risk.days_of_stock,
    "risk_level": risk.level,
    "reason": risk.reason,
    "top_transfer": asdict(risk.transfer_rank[0]) if risk.transfer_rank else None,
    "handoff_sw": handoff,
}

## 6. Mini evaluation on 3 synthetic cases

In [ ]:
rows = []
for item in EXAMPLES:
    predicted = extract_with_gemma(item["raw_input"], runtime)
    predicted_risk = calculate_stock_risk_with_transfers(predicted, inventory)
    rows.append(
        {
            "id": item["id"],
            "facility": predicted.facility,
            "item": predicted.item,
            "expected_risk": item["expected"]["risk_level"],
            "actual_risk": predicted_risk.level,
            "facility_ok": predicted.facility == item["expected"]["facility"],
            "item_ok": predicted.item.lower() == item["expected"]["item"].lower()
            or item["expected"]["item"].split()[0].lower() in predicted.item.lower(),
            "risk_ok": predicted_risk.level == item["expected"]["risk_level"],
        }
    )

rows

In [ ]:
passed = sum(1 for row in rows if row["facility_ok"] and row["item_ok"] and row["risk_ok"])
print(f"Smoke evaluation: {passed}/{len(rows)} examples fully matched")
print("Gemma preset:", GEMMA_PRESET)
print("Safety: Gemma extracts; AfyaFlow owns risk arithmetic, transfers, and handoff text.")

## Safety note

- Synthetic facility data only — no real patient records.
- Gemma extracts unstructured text; stock-risk math stays deterministic.
- Prototype thresholds (3 / 7 day bands) are demo assumptions, not county policy.